<a href="https://colab.research.google.com/github/SUBHA-coder/AI-AUTOMATED-BUG-FIXER-AND-AUTO-TESTING/blob/main/SentinelAI_Intelligent_Network_Anomaly_Detection_using_Isolation_Forestd20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pyod plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 944.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.6/413.6 kB 4.0 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import plotly.express as px
import plotly.graph_objects as go

np.random.seed(42)

In [3]:
normal_users = 5000

normal = pd.DataFrame({
    "packets_per_sec": np.random.normal(120,20,normal_users),
    "bytes_sent": np.random.normal(50000,9000,normal_users),
    "failed_logins": np.random.normal(1,1,normal_users),
    "unique_ports": np.random.normal(8,2,normal_users),
    "session_duration": np.random.normal(600,120,normal_users)
})

normal["label"]="Normal"

normal.head()

,packets_per_sec,bytes_sent,failed_logins,unique_ports,session_duration,label
0,129.934283,46186.162862,0.321505,7.713154,641.794350,Normal
1,117.234714,45919.273025,0.694501,7.934688,633.998831,Normal
2,132.953771,33839.211446,0.402619,8.128590,487.617619,Normal
3,150.460597,47029.188275,1.110418,9.893723,669.550107,Normal
4,115.316933,56595.461736,2.197179,6.505565,421.190079,Normal


In [4]:
attack = pd.DataFrame({
    "packets_per_sec": np.random.normal(900,80,120),
    "bytes_sent": np.random.normal(250000,40000,120),
    "failed_logins": np.random.normal(18,4,120),
    "unique_ports": np.random.normal(70,8,120),
    "session_duration": np.random.normal(40,15,120)
})

attack["label"]="Attack"

In [5]:
data = pd.concat([normal,attack],ignore_index=True)

data.sample(5)

,packets_per_sec,bytes_sent,failed_logins,unique_ports,session_duration,label
1764,115.078750,30884.064637,0.929954,7.839726,503.040580,Normal
982,100.660477,67922.333766,0.593971,10.414851,600.162396,Normal
3499,112.866537,59577.536321,0.797347,7.827230,611.259953,Normal
380,103.205563,44722.263660,3.349109,8.626247,460.428098,Normal
247,106.933415,35970.917045,1.631419,5.350277,704.891208,Normal


In [6]:
print("Dataset Shape :", data.shape)
print("\nClass Distribution")
print(data["label"].value_counts())

data.describe()

Dataset Shape : (5120, 6)

Class Distribution
label
Normal    5000
Attack     120
Name: count, dtype: int64


,packets_per_sec,bytes_sent,failed_logins,unique_ports,session_duration
count,5120.000000,5120.000000,5120.000000,5120.000000,5120.000000
mean,138.262799,54527.240832,1.406635,9.484272,584.798272
std,119.294087,31672.698697,2.806342,9.652198,143.688727
min,55.174653,14698.397735,-2.375579,0.287249,3.905828
25%,107.121154,44042.474935,0.357538,6.706286,508.915540
50%,120.874139,50123.548994,1.034299,8.084289,594.652478
75%,134.310083,56547.572950,1.738864,9.523178,675.588284
max,1092.369959,332477.398240,26.783892,89.929987,1033.362008


In [7]:
fig = px.scatter(
    data,
    x="packets_per_sec",
    y="bytes_sent",
    color="label",
    title="Network Traffic Distribution",
    opacity=0.75,
    template="plotly_dark"
)

fig.update_layout(height=600)
fig.show()

In [8]:
fig = px.histogram(
    data,
    x="failed_logins",
    color="label",
    nbins=40,
    barmode="overlay",
    opacity=0.7,
    title="Failed Login Attempts",
    template="plotly_dark"
)

fig.show()

In [9]:
import plotly.figure_factory as ff

corr = data.drop(columns=["label"]).corr()

fig = ff.create_annotated_heatmap(
    z=np.round(corr.values,2),
    x=list(corr.columns),
    y=list(corr.columns),
    colorscale="Viridis"
)

fig.update_layout(
    title="Feature Correlation Heatmap",
    template="plotly_dark",
    height=650
)

fig.show()

In [10]:
fig = px.scatter_3d(
    data,
    x="packets_per_sec",
    y="bytes_sent",
    z="failed_logins",
    color="label",
    opacity=0.75,
    template="plotly_dark",
    title="3D Network Traffic"
)

fig.update_layout(height=750)

fig.show()

In [11]:
fig = px.box(
    data,
    x="label",
    y="packets_per_sec",
    color="label",
    template="plotly_dark",
    title="Packets per Second Distribution"
)

fig.show()

In [12]:
features = data.drop(columns=["label"])

features.head()

,packets_per_sec,bytes_sent,failed_logins,unique_ports,session_duration
0,129.934283,46186.162862,0.321505,7.713154,641.794350
1,117.234714,45919.273025,0.694501,7.934688,633.998831
2,132.953771,33839.211446,0.402619,8.128590,487.617619
3,150.460597,47029.188275,1.110418,9.893723,669.550107
4,115.316933,56595.461736,2.197179,6.505565,421.190079


In [13]:
scaler = StandardScaler()

X = scaler.fit_transform(features)

print("Shape:", X.shape)

Shape: (5120, 5)


In [14]:
model = IsolationForest(
    n_estimators=300,
    contamination=0.023,      # ~120 attacks out of 5120 samples
    random_state=42,
    n_jobs=-1
)

model.fit(X)

IsolationForest(contamination=0.023, n_estimators=300, n_jobs=-1,
                random_state=42)

In [15]:
pred = model.predict(X)

data["prediction"] = np.where(pred == -1, "Attack", "Normal")

In [16]:
scores = model.decision_function(X)

data["anomaly_score"] = scores

data.head()

,packets_per_sec,bytes_sent,failed_logins,unique_ports,session_duration,label,prediction,anomaly_score
0,129.934283,46186.162862,0.321505,7.713154,641.794350,Normal,Normal,0.376783
1,117.234714,45919.273025,0.694501,7.934688,633.998831,Normal,Normal,0.383782
2,132.953771,33839.211446,0.402619,8.128590,487.617619,Normal,Normal,0.328736
3,150.460597,47029.188275,1.110418,9.893723,669.550107,Normal,Normal,0.343070
4,115.316933,56595.461736,2.197179,6.505565,421.190079,Normal,Normal,0.330454


In [17]:
print(pd.crosstab(
    data["label"],
    data["prediction"],
    margins=True
))

prediction  Attack  Normal   All
label                           
Attack         118       2   120
Normal           0    5000  5000
All            118    5002  5120


In [18]:
from sklearn.metrics import classification_report

true = data["label"].map({
    "Normal": 0,
    "Attack": 1
})

predicted = data["prediction"].map({
    "Normal": 0,
    "Attack": 1
})

print(classification_report(true, predicted))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5000
           1       1.00      0.98      0.99       120

    accuracy                           1.00      5120
   macro avg       1.00      0.99      1.00      5120
weighted avg       1.00      1.00      1.00      5120



In [19]:
fig = px.scatter(
    data,
    x="packets_per_sec",
    y="bytes_sent",
    color="prediction",
    hover_data=[
        "failed_logins",
        "unique_ports",
        "session_duration",
        "anomaly_score"
    ],
    title="Isolation Forest Predictions",
    template="plotly_dark"
)

fig.update_layout(height=650)

fig.show()

In [20]:
data.sort_values("anomaly_score").head(20)

,packets_per_sec,bytes_sent,failed_logins,unique_ports,session_duration,label,prediction,anomaly_score
5032,957.478379,280872.120011,25.016172,88.403600,50.152932,Attack,Attack,-0.047574
5103,985.628147,332477.398240,17.792410,86.776095,37.827588,Attack,Attack,-0.042949
5047,1043.190745,260738.475995,21.710366,78.698547,17.720572,Attack,Attack,-0.042417
5087,1066.725702,217664.446511,23.997151,89.929987,45.526447,Attack,Attack,-0.041531
5022,962.067516,315354.811131,24.577521,75.371821,49.300778,Attack,Attack,-0.040823
5045,878.483755,287806.189427,25.269473,82.557757,44.628853,Attack,Attack,-0.038527
5030,774.021650,278033.163993,17.131003,87.458142,43.684949,Attack,Attack,-0.035358
5021,824.295100,294967.071404,22.824458,75.228773,66.044857,Attack,Attack,-0.034130
5090,817.464134,259388.709314,23.921969,60.605176,16.720405,Attack,Attack,-0.032903
5018,977.753496,291512.277661,21.368631,77.636719,43.012598,Attack,Attack,-0.031503


In [21]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

cm = confusion_matrix(
    data["label"],
    data["prediction"],
    labels=["Normal", "Attack"]
)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=["Predicted Normal", "Predicted Attack"],
    y=["Actual Normal", "Actual Attack"],
    colorscale="Blues",
    showscale=True
)

fig.update_layout(
    title="Isolation Forest - Confusion Matrix",
    template="plotly_dark",
    height=600
)

fig.show()

In [22]:
actual = data["label"].value_counts().sort_index()
pred = data["prediction"].value_counts().sort_index()

comparison = pd.DataFrame({
    "Category": ["Attack", "Normal"],
    "Actual": [
        actual.get("Attack", 0),
        actual.get("Normal", 0)
    ],
    "Predicted": [
        pred.get("Attack", 0),
        pred.get("Normal", 0)
    ]
})

comparison

,Category,Actual,Predicted
0,Attack,120,118
1,Normal,5000,5002


In [23]:
fig = go.Figure()

fig.add_bar(
    x=comparison["Category"],
    y=comparison["Actual"],
    name="Actual",
)

fig.add_bar(
    x=comparison["Category"],
    y=comparison["Predicted"],
    name="AI Prediction",
)

fig.update_layout(
    title="Actual Labels vs AI Predictions",
    barmode="group",
    template="plotly_dark",
    height=600,
    xaxis_title="Class",
    yaxis_title="Number of Sessions"
)

fig.show()

In [24]:
fig = px.histogram(
    data,
    x="anomaly_score",
    color="label",
    nbins=80,
    opacity=0.7,
    marginal="box",
    template="plotly_dark",
    title="Anomaly Score Distribution"
)

fig.show()